### Preparation

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from tqdm import tqdm
from mltclass import QuantumNetwork
from mltclass.utils import load_dataset, split_dataset, get_accuracy, plot_history, plot_weights

In [ ]:
# Hyperparameters
num_epochs = 250
num_hidden = 30
batch_size = 512
learning_rate = 0.01
dataset = "MNIST" # Available (MNIST, Fashion, CIFAR)
rng = np.random.default_rng(2025)
torch.manual_seed(2025)
mode = "one_vs_one" # Available (one_vs_rest, one_vs_one)
balanced_dataset = False
use_bias_sigmoid = True
trainval_ratio = 0.8 # Ratio 80% training, 20% validation

### Dataset

In [ ]:
(X, Y), (XAll, YAll) = load_dataset(dataset, download = False)
(num_classes, num_models), (Xtrain, Ytrain), \
(Xval, Yval), (Xtest, Ytest) = split_dataset(X, Y, XAll, YAll, mode, balanced_dataset, trainval_ratio, rng, show_population = False)
train_loader = [DataLoader(TensorDataset(X, Y), batch_size = batch_size, shuffle = True) for X,Y in zip(Xtrain, Ytrain)]
val_loader = [DataLoader(TensorDataset(X, Y), batch_size = batch_size, shuffle = False) for X,Y in zip(Xval, Yval)]

### Training

In [ ]:
history_train = torch.zeros((num_models,num_epochs,2), dtype=torch.float64)
history_val = torch.zeros((num_models,num_epochs,2), dtype=torch.float64)
hidden_weights, output_weights, models, optimizer = [], [], [], []
for idx in tqdm(range(num_models)):
    models.append(QuantumNetwork(Xtrain[idx].shape[1], num_hidden, use_bias_sigmoid = use_bias_sigmoid)) # Model
    loss_function = torch.nn.BCELoss() # Loss
    optimizer.append(torch.optim.SGD(models[idx].parameters(), lr = learning_rate)) # Optimizers
    (tmptorchHid, tmptorchOut), history_train[idx,:,:], history_val[idx,:,:] = models[idx].train(train_loader[idx],val_loader[idx],\
                                                                               num_epochs,loss_function,optimizer[idx]) # Train
    # Uncomment for (hidden and output) weights tracking
    #hidden_weights.append(tmptorchHid) 
    #output_weights.append(tmptorchOut.squeeze(0)) # Squeeze 
#hidden_weights = torch.stack(hidden_weights)
#output_weights = torch.stack(output_weights)
del tmptorchHid, tmptorchOut

In [ ]:
plot_history(history_train, history_val, show_legend = False)

### Evaluation

In [ ]:
acc = get_accuracy(models, Xtest, Ytest, num_classes, num_models, mode) # Per-class and averaged accuracies
df = pd.DataFrame.from_dict(acc)
df = df.rename(index={df.index[-1]: 'Avg.'})
df.style.format("{:.2f}")#.set_caption("Accuracy")